In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

# Configuration esthétique pour les graphiques
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)


# 1. IMPORTATION ET NETTOYAGE DES DONNÉES

print("--- Étape 1 : Importation et Nettoyage ---")

# Chargement du fichier (adapter le nom du fichier CSV si nécessaire)
try:
    df = pd.read_csv("airplane_accidents_2023.csv")
    print("Données importées avec succès.")
except FileNotFoundError:
    # Génération d'un DataFrame fictif réaliste pour l'exemple si le fichier n'est pas local
    np.random.seed(42)
    dates = pd.date_range(start="1950-01-01", end="2023-12-31", periods=1000)
    df = pd.DataFrame(
        {
            "Date": dates,
            "Location": np.random.choice(
                ["USA", "France", "Brazil", "India", "Ocean"], size=1000
            ),
            "Operator": np.random.choice(
                ["Military", "Commercial", "Private"], size=1000
            ),
            "Aboard": np.random.randint(5, 300, size=1000),
        }
    )
    # Simuler des décès cohérents avec le nombre de personnes à bord
    df["Fatalities"] = df["Aboard"].apply(
        lambda x: int(x * np.random.beta(2, 2))
    )
    # Ajouter quelques valeurs manquantes pour la démonstration du nettoyage
    df.loc[df.sample(frac=0.02).index, "Fatalities"] = np.nan
    print("Fichier non trouvé. Utilisation d'un jeu de données simulé.")

# A. Conversion des dates au format approprié
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Year"] = df["Date"].dt.year
df["Decade"] = (df["Year"] // 10) * 10

# B. Traitement des valeurs manquantes
# Remplacement des décès manquants par la médiane ou suppression selon l'impact
fatalities_median = df["Fatalities"].median()
df["Fatalities"] = df["Fatalities"].fillna(fatalities_median)

# C. Calcul des colonnes clés : Taux de survie
# Taux de survie = (Personnes à bord - Décès) / Personnes à bord
df["Survivors"] = df["Aboard"] - df["Fatalities"]
# Éviter les divisions par zéro si 'Aboard' est égal à 0
df["Survival_Rate"] = np.where(
    df["Aboard"] > 0, df["Survivors"] / df["Aboard"], 0
)

print(df.info())



# 2. ANALYSE EXPLORATOIRE DES DONNÉES (EDA)

print("\n--- Étape 2 : Analyse Exploratoire ---")

total_accidents = len(df)
total_fatalities = df["Fatalities"].sum()
global_survival_rate = (
    df["Survivors"].sum() / df["Aboard"].sum() if df["Aboard"].sum() > 0 else 0
)

# Résumé sous forme de dictionnaire/table pour présentation claire
metrics_summary = {
    "Métrique": [
        "Nombre total d'accidents",
        "Nombre total de décès",
        "Taux de survie global",
    ],
    "Valeur": [
        f"{total_accidents}",
        f"{int(total_fatalities)}",
        f"{global_survival_rate:.2%}",
    ],
}
print(pd.DataFrame(metrics_summary).to_string(index=False))

# Analyse de la fréquence au fil du temps (par année)
accidents_per_year = df.groupby("Year").size()


# 3. ANALYSE STATISTIQUE

print("\n--- Étape 3 : Analyse Statistique ---")

# Statistiques descriptives clés avec SciPy / Pandas
mean_fatalities = np.mean(df["Fatalities"])
median_fatalities = np.median(df["Fatalities"])
std_fatalities = stats.tstd(df["Fatalities"])

print(f"Décès - Moyenne : {mean_fatalities:.2f}")
print(f"Décès - Médiane : {median_fatalities:.2f}")
print(f"Décès - Écart-type : {std_fatalities:.2f}")

# Test d'hypothèse : Comparaison du nombre de décès entre deux décennies (ex: 1970 vs 2010)
# Objectif : Valider si la sécurité aérienne a significativement réduit le nombre moyen de décès.
group_1970 = df[df["Decade"] == 1970]["Fatalities"]
group_2010 = df[df["Decade"] == 2010]["Fatalities"]

if len(group_1970) > 0 and len(group_2010) > 0:
    t_stat, p_val = stats.ttest_ind(group_1970, group_2010, equal_var=False)
    print(
        f"\nTest T de Student (Comparaison Décès 1970 vs 2010) :\n -> t-statistic = {t_stat:.4f}\n -> p-value = {p_val:.4f}"
    )
    if p_val < 0.05:
        print(
            "Le résultat est statistiquement significatif (p < 0.05). Il y a une différence réelle entre ces deux décennies."
        )
    else:
        print(
            "Le résultat n'est pas statistiquement significatif (p >= 0.05). Pas de différence prouvée."
        )


# 4. VISUALISATION DES DONNÉES

print("\n--- Étape 4 : Génération des Visualisations ---")

# Graphique 1 : Évolution du nombre d'accidents par année
plt.figure()
sns.lineplot(
    x=accidents_per_year.index,
    y=accidents_per_year.values,
    marker="o",
    color="crimson",
)
plt.title("Évolution du nombre d'accidents d'avion par année", fontsize=14)
plt.xlabel("Année")
plt.ylabel("Nombre d'accidents")
plt.tight_layout()
plt.show()

# Graphique 2 : Histogramme de la distribution des décès
plt.figure()
sns.histplot(df["Fatalities"], bins=30, kde=True, color="purple")
plt.title("Distribution du nombre de décès par accident", fontsize=14)
plt.xlabel("Nombre de décès")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()

# Graphique 3 : Boîte à moustaches (Boxplot) du taux de survie par décennie
plt.figure()
sns.boxplot(x="Decade", y="Survival_Rate", data=df, palette="Blues")
plt.title("Évolution du taux de survie par accident par décennie", fontsize=14)
plt.xlabel("Décennie")
plt.ylabel("Taux de survie (0 à 1)")
plt.tight_layout()
plt.show()